# 2. Load the pretrained model

The runner calls this after data preparation. We reuse the pretrained processor and tokenizer, and train all model parameters.

## Imports and required earlier steps

Choose the **Whisper (project .venv)** kernel. This cell imports the libraries and loads missing settings/prerequisites. Training stays disabled until you explicitly enable it.

In [ ]:
from pathlib import Path
import json
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Load settings when this notebook starts in a fresh kernel.
if "DATA_DIR" not in globals():
    root = Path.cwd().resolve()
    while not (root / "pyproject.toml").is_file():
        if root.parent == root:
            raise FileNotFoundError("Start the kernel inside the AI & ML project.")
        root = root.parent
    setup = json.loads((root / "notebooks/whisper/setup.ipynb").read_text())
    for cell in setup["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


## Load processor and model

The processor turns audio/text into model inputs. The model learns to predict the transcript.

In [ ]:
processor = WhisperProcessor.from_pretrained(MODEL_ID, local_files_only=LOCAL_FILES_ONLY)
processor.tokenizer.set_prefix_tokens(language=LANGUAGE, task="transcribe", predict_timestamps=False)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, local_files_only=LOCAL_FILES_ONLY)
model = model.to(DEVICE)
model.gradient_checkpointing_enable()  # Save activation memory during training.
model.config.use_cache = False
model.generation_config.language = LANGUAGE
model.generation_config.task = "transcribe"
model.generation_config.return_timestamps = False
model.generation_config.forced_decoder_ids = None

parameter_count = sum(parameter.numel() for parameter in model.parameters())
assert all(parameter.requires_grad for parameter in model.parameters())
print("Device:", DEVICE)
print("Trainable parameters:", f"{parameter_count:,}")
print("Encoder / decoder layers:", model.config.encoder_layers, "/", model.config.decoder_layers)